# 02 - Exploratory Data Analysis (EDA)

**What EDA is:** before you build any model, you look at the cleaned data with charts and simple stats to find patterns a model will need to capture — trend, seasonality, weekday effects, and the impact of weather/events/promos. EDA also catches anything the cleaning step missed.

**Why it matters for forecasting covers:** every pattern you confirm here becomes either a *feature* (something the model uses as an input) or a *baseline* (a simple rule to beat). If Saturdays are reliably 2.5x busier than Mondays, "day of week" has to be in the model. If local events add 50 covers, "is_local_event" has to be in the model too.

This notebook works through, in order: overall trend, day-of-week differences, monthly/seasonal pattern, weather effects, and event/holiday/promo effects.

## Step 1: Load the cleaned data

We built `covers_daily_clean.csv` in notebook 01. Load it and exclude days the restaurant was closed (`is_closed == 1`) — those are 0 covers by definition, not a demand signal, and would distort every average below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/covers_daily_clean.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

closed_days = df["is_closed"].sum()
print(f"Total rows: {len(df)}, closed days excluded from analysis: {closed_days}")

df_open = df[df["is_closed"] == 0].copy()
df_open.shape

## Step 2: Overall trend

**Concept — trend:** the slow, long-run direction of covers (growing, shrinking, flat) once you smooth out day-to-day noise. A 7-day rolling average is a simple smoother: it averages each day with the 6 before it, which cancels out weekday noise (a single rolling window always contains close to one of each weekday) and reveals the underlying level.

Look for: is the restaurant trending up/down over the year? Any sudden step-changes (a menu relaunch, a competitor opening, a renovation) that a manager would remember?

In [ ]:
df_open["covers_7d_avg"] = df_open["covers"].rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df_open["date"], df_open["covers"], alpha=0.3, label="Daily covers")
ax.plot(df_open["date"], df_open["covers_7d_avg"], color="darkred", linewidth=2, label="7-day rolling avg")
ax.set_title("Daily covers over time, with 7-day rolling average")
ax.set_xlabel("Date")
ax.set_ylabel("Covers")
ax.legend()
plt.tight_layout()
plt.show()

**In Sheets/Excel:** put `date` and `covers` in two columns, add a third column with `=AVERAGE(B2:B8)` dragged down (a 7-row trailing average), then chart all three as lines. Same idea, no code needed.

## Step 3: Day-of-week pattern

**Concept — weekday seasonality:** restaurants have a weekly cycle that's usually the single strongest driver of covers — much stronger than the yearly trend. Quantifying it precisely (not just "weekends are busier") is what lets you staff Tuesday differently from Saturday.

We group by `day_of_week` and look at both the average **and** the spread (std dev) — a day with high variance is less predictable and riskier to staff from an average alone.

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

dow_stats = (
    df_open.groupby("day_of_week")["covers"]
    .agg(mean="mean", std="std", min="min", max="max", n="count")
    .reindex(dow_order)
    .round(1)
)
dow_stats

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
df_open.boxplot(column="covers", by="day_of_week", ax=ax,
                 positions=range(7))
ax.set_xticks(range(7))
ax.set_xticklabels(dow_order, rotation=0)
ax.set_title("Covers distribution by day of week")
plt.suptitle("")
ax.set_xlabel("")
ax.set_ylabel("Covers")
plt.tight_layout()
plt.show()

**Read this table before moving on.** In this dataset, Saturday averages roughly 2.5x Monday, and Friday/Saturday/Sunday together are clearly a different regime from the Monday–Wednesday cluster. This is exactly the signal `day_of_week` and `is_weekend` are meant to capture as model features — and it means a naive "average of all days" baseline would be badly wrong on every single day.

## Step 4: Monthly / seasonal pattern

**Concept — seasonality:** a repeating pattern tied to the calendar, but on a longer cycle than a week — here, month of year (summer patio season, holiday months, slow post-holiday months). With a single year of data you can't actually tell "seasonality" apart from "that one year's trend" — a busy May could just as easily be a busy month generally as a one-off. With close to two full years now, we can check directly: does the same month look similarly busy in both years?

In [ ]:
month_stats = df_open.groupby("month")["covers"].mean().round(1)

fig, ax = plt.subplots(figsize=(9, 4))
month_stats.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Average covers by month")
ax.set_xlabel("Month")
ax.set_ylabel("Average covers")
ax.set_xticklabels(range(1, 13), rotation=0)
plt.tight_layout()
plt.show()

month_stats

**Does the pattern actually repeat?** Pivot the same monthly averages by year instead of collapsing across years — if a month's level is similar in both years, that's real seasonality; if it's wildly different, it's more likely noise or a one-off.

In [ ]:
year_month = df_open.groupby([df_open["date"].dt.year, df_open["date"].dt.month])["covers"].agg(["mean", "count"])
year_month_pivot = year_month["mean"].unstack(level=0).round(1)
year_month_pivot

**Reading this table:** most months land within a few points of each other year over year (January ~122 vs. ~129, May ~181 vs. ~183) — good evidence the monthly pattern is genuine seasonality, not a one-year fluke, with a modest overall uplift in the more recent year layered on top. The one exception is August, which jumps from the low 150s/160s up to 204 in the most recent year — but that's an artifact of the data cutting off just 2 days into that August, not a real seasonal shift; a 2-day average is noisy. Watch for this kind of edge effect whenever you're averaging a partial period.

## Step 5: Weather effects

**Concept:** weather can shift covers, but usually more mildly than day-of-week — and its effect can go either way (rain might keep walk-ins home, or push people toward a cozy indoor dinner instead of a backyard barbecue). Check it rather than assume it.

We look at a correlation (does higher temperature line up with more or fewer covers?) and a group-by on the categorical `condition` column.

In [ ]:
weather_corr = df_open[["covers", "temp_high_f", "precip_in"]].corr()["covers"]
print("Correlation with covers:")
print(weather_corr.round(3))

condition_stats = df_open.groupby("condition")["covers"].agg(mean="mean", n="count").round(1).sort_values("mean")
condition_stats

**How to read a correlation here:** values near 0 mean little/no linear relationship; near +1 or -1 mean a strong one. A correlation like 0.15–0.25 for temperature is a real but modest effect — worth including as a feature, not worth building the whole model around. If you saw something like 0.7+, that would be a much bigger deal and worth investigating further (and double-checking for a data error).

## Step 6: Holiday / event / promo effects

**Concept:** these are the flags most directly actionable by a manager — you *know in advance* whether next Thursday is a local event night or has a promo running, so if these move the needle, they belong in the model and in the weekly planning routine (Step 9 later in this project).

Compare the average covers on flagged days vs. unflagged days for each binary column.

In [ ]:
flags = ["is_holiday", "is_local_event", "is_promo", "is_weekend"]

flag_effects = pd.DataFrame({
    flag: df_open.groupby(flag)["covers"].mean().round(1)
    for flag in flags
}).T
flag_effects.columns = ["avg_covers_when_0", "avg_covers_when_1"]
flag_effects["lift"] = (flag_effects["avg_covers_when_1"] - flag_effects["avg_covers_when_0"]).round(1)
flag_effects

**In Sheets/Excel:** use `=AVERAGEIF(is_promo_range, 1, covers_range)` vs. `=AVERAGEIF(is_promo_range, 0, covers_range)` to get the same two numbers for each flag, no pivot table required.

## Recap

- **Trend:** the 7-day rolling average shows the overall level and whether it's drifting up or down across the data.
- **Day of week is the dominant pattern** — the gap between quiet weekdays and Fri/Sat/Sun is large enough that it has to anchor both the baseline rules and the model.
- **Monthly seasonality holds up across two separate years** now that the data spans close to 24 months, not just one cycle — a real hump rising into spring/summer, not a one-year fluke.
- **Weather has a real but modest effect** — worth including as a feature, not a headline driver.
- **Local events show a clear positive lift; holidays and promos are closer to a wash** once you compare raw group averages across the full dataset — a reminder that a simple "average when flag=1 vs. flag=0" comparison doesn't control for which weekdays those flagged days happen to fall on. The regression model in notebook 05 controls for day-of-week directly, which is exactly why it's a better tool for isolating each feature's real effect than this raw comparison.

**Next up:** we'll turn the day-of-week and rolling-average ideas from this notebook into baseline forecasting rules — now including a true same-date-last-year baseline, since two years of history makes that possible for the first time — and measure their error with MAE and MAPE, so we have a number the real model has to beat.